# Capstone 3 — Hangman

Hangman is a state machine with a small public surface. The secret word stays private. The caller proposes a letter and receives the masked word, the remaining lives, and the set of misses. A repeated guess is rejected rather than charged twice.

The demonstration plays a fixed sequence so the notebook has a visible game. `play_interactive` is included for a terminal session; it is not called here, because `input` would block the notebook.


In [1]:
from dataclasses import dataclass, field

MAX_LIVES = 6


@dataclass
class Hangman:
    secret: str
    lives: int = MAX_LIVES
    found: set[str] = field(default_factory=set)
    missed: set[str] = field(default_factory=set)

    def __post_init__(self):
        self.secret = self.secret.lower().strip()
        if not self.secret.isalpha():
            raise ValueError("secret must contain only letters")

    def masked(self) -> str:
        return " ".join(ch if ch in self.found else "_" for ch in self.secret)

    def guess(self, letter: str) -> str:
        if self.over:
            return "game over"
        cleaned = letter.lower().strip()
        if len(cleaned) != 1 or not cleaned.isalpha():
            return "enter one letter"
        if cleaned in self.found or cleaned in self.missed:
            return "already guessed"
        if cleaned in self.secret:
            self.found.add(cleaned)
            return "hit" if not self.won else "won"
        self.missed.add(cleaned)
        self.lives -= 1
        return "won" if self.won else "lost" if self.lost else "miss"

    @property
    def won(self) -> bool:
        return set(self.secret) <= self.found

    @property
    def lost(self) -> bool:
        return self.lives == 0 and not self.won

    @property
    def over(self) -> bool:
        return self.won or self.lost

    def status(self) -> str:
        return f"{self.masked()}   lives={self.lives}   misses={''.join(sorted(self.missed)) or '-'}"


def play_interactive(secret: str) -> None:
    game = Hangman(secret)
    while not game.over:
        print(game.status())
        game.guess(input("letter: "))
    print(game.status(), "won" if game.won else "lost")


game = Hangman("python")
for letter in ["a", "p", "p", "y", "t", "z", "h", "o", "n"]:
    result = game.guess(letter)
    print(f"{letter}: {result:16} {game.status()}")
print("won:", game.won)


a: miss             _ _ _ _ _ _   lives=5   misses=a
p: hit              p _ _ _ _ _   lives=5   misses=a
p: already guessed  p _ _ _ _ _   lives=5   misses=a
y: hit              p y _ _ _ _   lives=5   misses=a
t: hit              p y t _ _ _   lives=5   misses=a
z: miss             p y t _ _ _   lives=4   misses=az
h: hit              p y t h _ _   lives=4   misses=az
o: hit              p y t h o _   lives=4   misses=az
n: won              p y t h o n   lives=4   misses=az
won: True


## What to notice

`won` is derived from the set of found letters, not stored as a separate flag that could drift from the board. The repeated `p` is reported and does not reduce the life count. That is the difference between a demo and a game a learner can extend with a word list or a score table.
